# Sběr a spojení dat z Web Robots

Tento notebook připravuje dataset kampaní spuštěných v roce 2025, se kterým pracuje [hlavní notebook](../main.ipynb). Data pocházejí od [Web Robots](https://webrobots.io/kickstarter-datasets/), kteří přibližně jednou měsíčně stahují údaje o projektech na Kickstarteru.

Snapshot není přírůstkem ani úplnou databází. Jednotlivé snapshoty jsou téměř úplné, vždy v nich však část projektů chybí a tentýž projekt se zpravidla vyskytuje ve více snapshotech. Samotný snapshot z března 2026 například neobsahuje 3 627 projektů spuštěných v roce 2025, které se vyskytují ve starších snapshotech (viz [sekce 3](#s3)). Z toho důvodu jsme nepoužili pouze nejnovější snapshot, ale spojili jsme pět snapshotů (12. 6., 14. 7. a 18. 12. 2025, 12. 1. a 12. 3. 2026) podle ID projektu a pro každý projekt ponechali řádek z nejnovějšího z nich.

Výsledkem je 29 599 různých projektů, z toho 26 642 ve stavu `successful` nebo `failed`. Původně jsme pracovali se třemi snapshoty, další dva přidaly 53 známých výsledků. V prosinci je dokončených kampaní méně než v ostatních měsících. Může jít buď o sezónnost, nebo o neúplnost sběru.

Ke spuštění jsou potřeba rozbalené CSV soubory snapshotů ve složce `_data/raw/`, které kvůli své velikosti (celkem více než 9 GB) nejsou součástí projektu, ale je možné je stáhnout na následujících odkazech. Pokud ve složce chybí, první buňka je stáhne a rozbalí sama (ke stažení přibližně 1,8 GB, po rozbalení přes 9 GB). Notebook používá pouze standardní knihovnu Pythonu a IPython. Výsledek ukládá do čtyř souborů ve složce `_data/prepared/` a při opakovaném spuštění je přepíše.

Konkrétní použité snapshoty:
- [2025-06-12](https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2025-06-12T03_20_15_713Z.zip)
- [2025-07-14](https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2025-07-14T06_25_27_218Z.zip)
- [2025-12-18](https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2025-12-18T03_20_24_296Z.zip)
- [2026-01-12](https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2026-01-12T09_37_51_016Z.zip)
- [2026-03-12](https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2026-03-12T03_20_26_556Z.zip)

*Rozbalené vložit do složky `_data/raw/`*

In [37]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import csv
import zipfile
from html import escape
from urllib.request import urlretrieve
from IPython.display import HTML, display

def display_table(headers, rows):
    header = "".join(f"<th>{escape(str(value))}</th>" for value in headers)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

# Funguje při spuštění z kořene projektu i ze složky notebooks.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "_data").is_dir() else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "_data" / "raw"
SNAPSHOT_DATES = ("2025-06-12", "2025-07-14", "2025-12-18", "2026-01-12", "2026-03-12")
SNAPSHOT_NAMES = (
    "Kickstarter_2025-06-12T03_20_15_713Z",
    "Kickstarter_2025-07-14T06_25_27_218Z",
    "Kickstarter_2025-12-18T03_20_24_296Z",
    "Kickstarter_2026-01-12T09_37_51_016Z",
    "Kickstarter_2026-03-12T03_20_26_556Z",
)
SNAPSHOT_URL = "https://s3.amazonaws.com/weruns/forfun/Kickstarter/"

# Chybějící snapshoty stáhneme od Web Robots a rozbalíme.
# Soubory i složky se přejmenují až po úplném stažení a rozbalení.
RAW_DIR.mkdir(parents=True, exist_ok=True)
for name in SNAPSHOT_NAMES:
    folder = RAW_DIR / name
    archive = RAW_DIR / f"{name}.zip"
    if folder.is_dir():
        continue
    if not archive.exists():
        print(f"Stahuji {archive.name}…")
        urlretrieve(SNAPSHOT_URL + archive.name, archive.with_suffix(".tmp"))
        archive.with_suffix(".tmp").rename(archive)
    print(f"Rozbaluji {archive.name}…")
    with zipfile.ZipFile(archive) as zip_file:
        zip_file.extractall(RAW_DIR / f"{name}.unzip")
    (RAW_DIR / f"{name}.unzip").rename(folder)

print("Složka dat:", RAW_DIR.relative_to(PROJECT_ROOT))

Složka dat: _data/raw


<a id="s1"></a>
## 1. Načtení snapshotů

Každý snapshot se skládá z více CSV souborů. V této části z nich načítáme pouze údaje potřebné pro přehled (ID, datum spuštění, stav), samotné soubory neměníme. Pokud se stejné ID v jednom snapshotu vyskytuje vícekrát, zatím ponecháváme první výskyt a rozdíly mezi duplicitami pouze vypisujeme. Způsob zpracování duplicit ve výsledném datasetu popisuje [sekce 7](#s7).

In [38]:
csv.field_size_limit(100_000_000)
snapshots = {}

for date in SNAPSHOT_DATES:
    folders = sorted(path for path in RAW_DIR.glob(f"Kickstarter_{date}T*") if path.is_dir())
    if len(folders) != 1:
        raise ValueError(f"Pro {date} očekávám jednu rozbalenou složku, našel jsem: {folders}")
    files = sorted(folders[0].glob("*.csv"))
    if not files:
        raise FileNotFoundError(f"Ve složce {folders[0]} nejsou CSV soubory")

    projects = {}
    rows = 0
    missing_ids = 0
    conflicts = 0
    columns = None
    for file in files:
        with file.open(encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream)
            if columns is None:
                columns = reader.fieldnames
            elif columns != reader.fieldnames:
                raise ValueError(f"Odlišné sloupce v souboru {file}")
            for row in reader:
                rows += 1
                project_id = row["id"].strip()
                if not project_id:
                    missing_ids += 1
                    continue
                record = (row["launched_at"], row["state"])
                if project_id in projects and projects[project_id] != record:
                    conflicts += 1
                projects.setdefault(project_id, record)

    snapshots[date] = {"projects": projects, "rows": rows, "files": len(files),
                       "missing_ids": missing_ids, "conflicts": conflicts, "columns": columns}

summary = []
for date, snapshot in snapshots.items():
    count = len(snapshot["projects"])
    duplicates = snapshot["rows"] - snapshot["missing_ids"] - count
    summary.append((date, snapshot["files"], snapshot["rows"], count, duplicates,
                    snapshot["missing_ids"], snapshot["conflicts"]))
display_table(("Snapshot", "CSV", "Řádky", "Unikátní ID", "Opakování ID", "Bez ID", "Konfliktní opakování"), summary)

Snapshot,CSV,Řádky,Unikátní ID,Opakování ID,Bez ID,Konfliktní opakování
2025-06-12,82,259976,206323,53653,0,4
2025-07-14,81,257692,204301,53391,0,4
2025-12-18,84,266378,208465,57913,0,2
2026-01-12,84,266279,206872,59407,0,4
2026-03-12,85,268539,209625,58914,0,2


<a id="s2"></a>
## 2. Projekty spuštěné v roce 2025

Rok určujeme podle data spuštění (`launched_at`). Nulové, chybějící nebo neplatné časové údaje počítáme zvlášť. Stav odpovídá dni sběru, stav `live` proto ještě neznamená konečný výsledek.

In [39]:
def launch_year(timestamp):
    try:
        value = int(timestamp)
        if value <= 0:
            return None
        return datetime.fromtimestamp(value, tz=timezone.utc).year
    except (ValueError, TypeError, OverflowError, OSError):
        return None

projects_2025 = {}
for date, snapshot in snapshots.items():
    projects = snapshot["projects"]
    selected = {project_id: state for project_id, (launched_at, state) in projects.items()
                if launch_year(launched_at) == 2025}
    projects_2025[date] = selected

rows_2025 = []
for date, projects in projects_2025.items():
    states = Counter(projects.values())
    rows_2025.append((date, len(projects), states["successful"], states["failed"],
                      states["live"], states["canceled"]))
display_table(("Snapshot", "Projekty 2025", "Úspěšné", "Neúspěšné", "Live", "Zrušené"), rows_2025)

Snapshot,Projekty 2025,Úspěšné,Neúspěšné,Live,Zrušené
2025-06-12,13280,6877,2649,3264,490
2025-07-14,15588,8613,3389,2979,607
2025-12-18,27117,16885,6739,2435,1058
2026-01-12,26978,17667,7098,1122,1091
2026-03-12,25972,17399,7488,0,1085


<a id="s3"></a>
## 3. Spojení snapshotů

Projekty spojujeme podle ID a stav přebíráme z nejnovějšího snapshotu, ve kterém se projekt vyskytuje. Srovnání s nejnovějším snapshotem z března 2026 ukazuje, kolik projektů máme k dispozici pouze díky starším sběrům.

In [40]:
march_ids = set(projects_2025["2026-03-12"])
all_ids = set().union(*(set(projects) for projects in projects_2025.values()))
latest_state = {}
for date in SNAPSHOT_DATES:
    latest_state.update(projects_2025[date])

extra_ids = all_ids - march_ids
all_states = Counter(latest_state.values())
extra_states = Counter(latest_state[project_id] for project_id in extra_ids)
display_table(("Stav", "Všechny snapshoty", "Navíc oproti březnu"),
              [(state, all_states[state], extra_states[state]) for state in sorted(all_states)])
print(f"Celkem unikátních projektů: {len(all_ids):,}; navíc oproti březnu: {len(extra_ids):,}")

Stav,Všechny snapshoty,Navíc oproti březnu
canceled,1187,102
failed,7806,318
live,1770,1770
successful,18836,1437


Celkem unikátních projektů: 29,599; navíc oproti březnu: 3,627


<a id="s4"></a>
## 4. Kampaně podle měsíce spuštění

Tabulka uvádí nejnovější známé stavy podle měsíce spuštění. Stav `live` označuje projekty, jejichž konečný výsledek z dostupných dat neznáme. Prosinec má méně kampaní se známým výsledkem.

In [41]:
month_names = ("leden", "únor", "březen", "duben", "květen", "červen", "červenec",
               "srpen", "září", "říjen", "listopad", "prosinec")
latest_month = {}
for date in SNAPSHOT_DATES:
    for project_id in projects_2025[date]:
        timestamp, _ = snapshots[date]["projects"][project_id]
        latest_month[project_id] = datetime.fromtimestamp(int(timestamp), tz=timezone.utc).month

monthly = {month: Counter() for month in range(1, 13)}
for project_id, state in latest_state.items():
    monthly[latest_month[project_id]][state] += 1

monthly_rows = []
for month, name in enumerate(month_names, 1):
    states = monthly[month]
    other = sum(states.values()) - states["successful"] - states["failed"] - states["live"]
    monthly_rows.append((name, states["successful"], states["failed"],
                         states["successful"] + states["failed"], states["live"], other, sum(states.values())))
display_table(("Měsíc", "Úspěšné", "Neúspěšné", "Známý výsledek", "Live", "Ostatní", "Celkem"), monthly_rows)

Měsíc,Úspěšné,Neúspěšné,Známý výsledek,Live,Ostatní,Celkem
leden,1404,666,2070,0,92,2162
únor,1581,630,2211,0,73,2284
březen,1638,617,2255,0,113,2368
duben,1596,650,2246,62,94,2402
květen,1587,699,2286,349,128,2763
červen,1641,717,2358,510,129,2997
červenec,1550,686,2236,259,96,2591
srpen,1472,587,2059,0,89,2148
září,1790,670,2460,0,77,2537
říjen,2086,714,2800,35,106,2941


<a id="s5"></a>
## 5. Příspěvek jednotlivých snapshotů

Graf skládá projekty spuštěné v roce 2025 od nejnovějšího snapshotu k nejstaršímu. První pruh zahrnuje všechny projekty ze snapshotu z března 2026, každý další pouze projekty, které se v novějších snapshotech nevyskytly. Zahrnuty jsou všechny stavy včetně `live` a zrušených kampaní.

In [42]:
seen = set()
contributions = []
for date in reversed(SNAPSHOT_DATES):
    ids = set(projects_2025[date])
    new_ids = ids - seen
    contributions.append((date, len(new_ids), len(ids), len(ids & seen)))
    seen.update(ids)

max_count = max(count for _, count, _, _ in contributions)
colors = ("#4169a8", "#5490a3", "#66a182", "#c49a56", "#bb726b")
bars = []
for (date, count, total, overlap), color in zip(contributions, colors):
    width = 100 * count / max_count if max_count else 0
    bars.append(
        f'<div style="margin: 10px 0"><strong>{date}</strong> '
        f'<span style="color:#555">— nově {count:,} / celkem {total:,}</span>'
        f'<div style="background:#eee; width:100%; max-width:620px; height:20px; margin-top:4px">'
        f'<div style="background:{color}; width:{width:.2f}%; height:20px"></div></div></div>'
    )
display(HTML("".join(bars)))
display_table(("Snapshot", "Nově přidaná ID", "ID již v novějších snapshotech", "ID ve snapshotu celkem"),
              [(date, new, overlap, total) for date, new, total, overlap in contributions])
print(f"Celkem unikátních projektů: {len(seen):,}")

Snapshot,Nově přidaná ID,ID již v novějších snapshotech,ID ve snapshotu celkem
2026-03-12,25972,0,25972
2026-01-12,1079,25899,26978
2025-12-18,610,26507,27117
2025-07-14,1350,14238,15588
2025-06-12,588,12692,13280


Celkem unikátních projektů: 29,599


<a id="s6"></a>
## 6. Přínos dvou dodatečných snapshotů

Původně jsme pracovali se třemi snapshoty (červen a prosinec 2025, březen 2026), následně jsme doplnili červenec 2025 a leden 2026. V této části porovnáváme, kolik známých výsledků tím přibylo. Může jít o zcela nový projekt nebo o projekt, který byl v původních třech snapshotech zachycen pouze ve stavu `live`.

In [43]:
original_dates = ("2025-06-12", "2025-12-18", "2026-03-12")
original_state = {}
for date in original_dates:
    original_state.update(projects_2025[date])
finished = {"successful", "failed"}
original_finished = {pid for pid, state in original_state.items() if state in finished}
current_finished = {pid for pid, state in latest_state.items() if state in finished}
new_finished = current_finished - original_finished
lost_finished = original_finished - current_finished
display_table(("Srovnání", "Počet projektů"), [
    ("Známý výsledek: původní tři snapshoty", len(original_finished)),
    ("Známý výsledek: všech pět snapshotů", len(current_finished)),
    ("Nově známý výsledek", len(new_finished)),
    ("Dříve známý, nyní jiný stav", len(lost_finished)),
])

Srovnání,Počet projektů
Známý výsledek: původní tři snapshoty,26589
Známý výsledek: všech pět snapshotů,26642
Nově známý výsledek,53
"Dříve známý, nyní jiný stav",0


<a id="s7"></a>
## 7. Sestavení datasetu

Pro každý projekt spuštěný v roce 2025 ponecháváme celý původní řádek z nejnovějšího snapshotu, ve kterém se projekt vyskytuje. Zachováváme všechny stavy včetně `live` a `canceled`, výběr pro model probíhá až v hlavním notebooku. Pokud se stejné ID v jednom snapshotu vyskytuje vícekrát, vybíráme první výskyt.

K řádku doplňujeme sloupce o původu záznamu: první a poslední snapshot, ve kterém se projekt vyskytl, počet a seznam snapshotů, zdrojový soubor a příznak konfliktu. Původní hodnoty (texty, stav, částky) neměníme.

In [44]:
# Načítáme plné řádky jen pro projekty vybrané v předchozím průzkumu.
# Snapshoty procházíme od nejstaršího, takže novější záznam přepíše starší.
records = {}
provenance = {}
conflicted_ids = set()
for date in SNAPSHOT_DATES:
    folder = next(path for path in RAW_DIR.glob(f"Kickstarter_{date}T*") if path.is_dir())
    seen_in_snapshot = set()
    for file in sorted(folder.glob("*.csv")):
        with file.open(encoding="utf-8-sig", newline="") as stream:
            for row in csv.DictReader(stream):
                project_id = row["id"].strip()
                if project_id not in projects_2025[date]:
                    continue
                if project_id in seen_in_snapshot:
                    if (row["launched_at"], row["state"]) != (records[project_id]["launched_at"], records[project_id]["state"]):
                        conflicted_ids.add(project_id)
                    continue
                seen_in_snapshot.add(project_id)
                records[project_id] = row
                if project_id not in provenance:
                    provenance[project_id] = {"dates": []}
                provenance[project_id]["dates"].append(date)
                provenance[project_id]["file"] = file.name
    assert seen_in_snapshot == set(projects_2025[date]), f"Nesouhlasí načtená ID pro {date}"
    print(f"{date}: načteno {len(seen_in_snapshot):,} unikátních projektů z roku 2025")

assert set(records) == all_ids
print(f"Celkem: {len(records):,} projektů; konfliktní stav nebo datum u {len(conflicted_ids):,} ID")

2025-06-12: načteno 13,280 unikátních projektů z roku 2025
2025-07-14: načteno 15,588 unikátních projektů z roku 2025
2025-12-18: načteno 27,117 unikátních projektů z roku 2025
2026-01-12: načteno 26,978 unikátních projektů z roku 2025
2026-03-12: načteno 25,972 unikátních projektů z roku 2025
Celkem: 29,599 projektů; konfliktní stav nebo datum u 7 ID


<a id="s8"></a>
## 8. Uložení

Dataset ukládáme do několika CSV souborů o velikosti nejvýše 50 MB, každý s vlastní hlavičkou. Obsahují původních 42 sloupců a 6 sloupců o původu záznamu. Vnořené údaje (např. kategorie) zůstávají v původní podobě JSON textu. Soubory lze kdykoli znovu vytvořit spuštěním notebooku.

In [45]:
import io

PREPARED_DIR = PROJECT_ROOT / "_data" / "prepared"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)
PART_PREFIX = "kickstarter_projects_2025_part_"
MAX_PART_BYTES = 50_000_000  # 50 MB včetně hlavičky

original_columns = snapshots[SNAPSHOT_DATES[0]]["columns"]
for date in SNAPSHOT_DATES:
    if snapshots[date]["columns"] != original_columns:
        raise ValueError(f"Snapshot {date} má jiné sloupce; před uložením je potřeba je sjednotit.")
extra_columns = ["snapshot_first_seen", "snapshot_last_seen", "snapshot_count", "snapshot_dates",
                 "snapshot_source_file", "snapshot_has_state_or_launch_conflict"]
columns = original_columns + extra_columns
if set(original_columns) & set(extra_columns):
    raise ValueError("Původní sloupce kolidují s metadaty snapshotů.")

# Celý CSV záznam (i případné odřádkování v textu) musí zůstat v jednom souboru.
buffer = io.StringIO(newline="")
writer = csv.DictWriter(buffer, fieldnames=columns)
def encoded_row(row=None):
    buffer.seek(0)
    buffer.truncate(0)
    writer.writeheader() if row is None else writer.writerow(row)
    return buffer.getvalue().encode("utf-8")

header = encoded_row()
part_paths = []
temporary_paths = []
stream = None
try:
    for project_id in sorted(records, key=int):
        info = provenance[project_id]
        row = {**records[project_id],
               "snapshot_first_seen": info["dates"][0],
               "snapshot_last_seen": info["dates"][-1],
               "snapshot_count": len(info["dates"]),
               "snapshot_dates": "|".join(info["dates"]),
               "snapshot_source_file": info["file"],
               "snapshot_has_state_or_launch_conflict": project_id in conflicted_ids}
        data = encoded_row(row)
        if len(header) + len(data) > MAX_PART_BYTES:
            raise ValueError(f"Projekt {project_id} je příliš velký pro jeden soubor.")
        if stream is None or stream.tell() + len(data) > MAX_PART_BYTES:
            if stream is not None:
                stream.close()
            path = PREPARED_DIR / f"{PART_PREFIX}{len(part_paths) + 1:03d}.csv"
            temporary = path.with_suffix(".csv.tmp")
            part_paths.append(path)
            temporary_paths.append(temporary)
            stream = temporary.open("wb")
            stream.write(header)
        stream.write(data)
    if stream is not None:
        stream.close()
        stream = None
    for temporary, path in zip(temporary_paths, part_paths):
        temporary.replace(path)
    for old_part in PREPARED_DIR.glob(f"{PART_PREFIX}[0-9][0-9][0-9].csv"):
        if old_part not in part_paths:
            old_part.unlink()
finally:
    if stream is not None:
        stream.close()
    for temporary in temporary_paths:
        temporary.unlink(missing_ok=True)

for path in part_paths:
    print(f"{path.name}: {path.stat().st_size / 1_000_000:.2f} MB")

kickstarter_projects_2025_part_001.csv: 50.00 MB
kickstarter_projects_2025_part_002.csv: 49.99 MB
kickstarter_projects_2025_part_003.csv: 50.00 MB
kickstarter_projects_2025_part_004.csv: 39.28 MB


<a id="s9"></a>
## 9. Kontrola uložených dat

Všechny části znovu načteme a ověříme počet řádků a unikátních ID, rozložení stavů a měsíců a úplnost základních údajů. Za chybějící zde považujeme pouze prázdné řetězce. Jiné podoby prázdných hodnot (např. `null` u videa) zpracovává hlavní notebook.

In [46]:
saved_ids = set()
saved_states = Counter()
saved_months = Counter()
missing = Counter()
saved_rows = 0
missing_fields = ("id", "name", "blurb", "category", "goal", "country", "launched_at", "state")
for path in part_paths:
    assert path.stat().st_size <= MAX_PART_BYTES
    with path.open(encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == original_columns + extra_columns
        for row in reader:
            saved_rows += 1
            saved_ids.add(row["id"])
            saved_states[row["state"]] += 1
            saved_months[datetime.fromtimestamp(int(row["launched_at"]), tz=timezone.utc).month] += 1
            for field in missing_fields:
                if not row[field].strip():
                    missing[field] += 1

assert saved_rows == len(saved_ids) == len(records)
assert saved_ids == set(records)
assert saved_states == Counter(latest_state.values())
assert all(launch_year(row["launched_at"]) == 2025 for row in records.values())
display_table(("Kontrola", "Hodnota"), [
    ("Řádky", saved_rows), ("Unikátní ID", len(saved_ids)),
    ("Duplicitní ID", saved_rows - len(saved_ids)),
    ("Konfliktní stav nebo datum ve snapshotu", len(conflicted_ids)),
])
display_table(("Stav", "Počet projektů"), saved_states.most_common())
display_table(("Sloupec", "Prázdné hodnoty"), [(field, missing[field]) for field in missing_fields])

Kontrola,Hodnota
Řádky,29599
Unikátní ID,29599
Duplicitní ID,0
Konfliktní stav nebo datum ve snapshotu,7


Stav,Počet projektů
successful,18836
failed,7806
live,1770
canceled,1187


Sloupec,Prázdné hodnoty
id,0
name,0
blurb,1
category,0
goal,0
country,0
launched_at,0
state,0


## Závěr a omezení

Výsledkem je pracovní dataset se všemi stavy a původními hodnotami, bez čištění pro účely modelování. Výběr dokončených kampaní, rozdělení dat a příprava příznaků probíhají v [hlavním notebooku](../main.ipynb).

Omezení:
- Nejnovější řádek nemusí popisovat projekt v den spuštění. Částky a počty podporovatelů se pro předpověď nehodí a i texty nebo další údaje se mohly později změnit.
- Kickstarter umožňuje i příspěvky po skončení kampaně ([Late Pledges](https://features.kickstarter.com/late-pledges)), vybraná částka proto nemusí odpovídat stavu v den ukončení.
- U 7 projektů se snapshoty rozcházejí ve stavu nebo datu spuštění.
- Výsledky se týkají pouze projektů zachycených Web Robots a nemusí představovat úplné pokrytí Kickstarteru.